In [ ]:
import pandas
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import SGDRegressor
from sklearn.neighbors import KNeighborsRegressor

In [ ]:
dataset = pandas.read_csv('/kaggle/input/car-price-prediction/CarPrice_Assignment.csv')
dataset

In [ ]:
print(dataset.shape)
dataset.columns

In [ ]:
def absolute(x):
    return np.abs(x)
numeric_dataset = dataset.select_dtypes(np.number)
CorrMatrix = numeric_dataset.corr()
CorrMatrix['price'].sort_values(key = absolute, ascending = False)

'''
To Drop

carheight
car_ID   
peakrpm  
symboling
stroke   
compressionratio
'''

In [ ]:
target = dataset['price'].copy()
data = dataset.drop(['carheight', 'car_ID', 'peakrpm', 'symboling', 'stroke', 'compressionratio', 'price'], axis = 1)

In [ ]:
data

In [ ]:
companies_list = []
models_list = []
for name in dataset['CarName']:
    company = ''
    model = ''
    for i in range(len(name)):
        if name[i] == ' ':
            for j in range(i+1, len(name)):
                if name[j] != ' ':
                    model = model + name[j]
            break
        company = company + name[i]
    companies_list.append(company)
    models_list.append(model)
companies_list[:3], models_list[:3]

In [ ]:
data['company'] = companies_list
data['model'] = models_list

In [ ]:
data = data.drop('CarName', axis = 1)
data

# Numerating
Numerating categorical values in such a way that increases the correlation between the numeric values and the price, to do that I made a dictionary that contain the mean price of cars for every category sorted by price, then mapping the column of the categorical feature to this dictionary.

In [ ]:
temp = data.select_dtypes('object')
for feature in temp.columns:
    dic = pandas.concat([temp[feature], target], axis = 1).groupby(feature).mean().sort_values(by = 'price')
    mp = {}
    p = 0
    for it in dic.index:
        mp[it] = p
        p += 1
    temp[feature] = temp[feature].map(mp)

In [ ]:
data[data.select_dtypes('object').columns] = temp
data

In [ ]:
for it in data.columns:
    mx = data[it].max()
    mn = data[it].min()
    data[it] = (data[it] - mn) / (mx - mn)
data

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(data, target, test_size = 0.3, random_state = 10)

In [ ]:
param_grid = [
    [i for i in range(1, 15)],
    [i for i in range(1, 10)],
    [i for i in range(1, 10)]
]
mx = 0
best = [0 for i in range(len(param_grid))]

In [ ]:
def dfs(p, lis):
    global mx
    if p == len(param_grid):
        model = RandomForestRegressor(n_estimators = lis[0], max_depth = lis[1], max_features = lis[2], random_state = 18)
        model.fit(X_train, y_train)
        acc = model.score(X_test, y_test)
        if acc > mx:
            mx = acc
            for i in range(len(lis)):
                best[i] = lis[i]
        return
    for i in range(len(param_grid[p])):
        lis.append(param_grid[p][i])
        dfs(p+1, lis)
        lis.pop(-1)
dfs(0, [])
best, mx

In [ ]:
print(best, mx)
RF_reg = RandomForestRegressor(n_estimators = best[0], max_depth = best[1], max_features = best[2], random_state = 18)
RF_reg.fit(X_train, y_train)
RF_reg.score(X_test, y_test)